# Kavach: host Gnani Evon on Kaggle (free 2× T4)

Serves **gnani/gnani-evon-v3.3-30B-A3B** (community GGUF, IQ4_XS, 18.4 GB) with llama.cpp's
OpenAI-compatible server, behind a Cloudflare quick tunnel protected by an API key.

**Before running:** Settings (right panel) → Accelerator **GPU T4 x2**, and **Internet on**
(needs a phone-verified Kaggle account). Then *Run All*. First run: ~30 min build + ~5 min download.

The last cell prints the two lines to put in `kavach/.env`. Kaggle sessions stop after ~12 h or when idle;
re-run to get a new URL.

**Skip the 30-minute build next time:**
1. After a successful build, click **Save Version**. **Save & Run All (Commit)** reliably keeps
   `/kaggle/working/llama-bin` as the notebook's output; it re-runs in the background and does not touch your
   open session. *Quick Save* may also keep it, but this hasn't been checked.
2. In the next session, use **Add Input → Your Work** to attach this notebook's saved output.
3. *Run All*. Step 1 finds the saved `llama-server` and skips the compile, so only the ~5 min model download remains.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build llama.cpp with CUDA (T4 = compute capability 7.5, building only that is much faster)

In [ ]:
%%bash
set -e
mkdir -p /tmp/llama-bin

# Reuse a server built in an earlier session, if you attached that session's output (see the note at the top)
SAVED=$(ls /kaggle/input/*/llama-bin/llama-server 2>/dev/null | head -1 || true)
if [ -n "$SAVED" ]; then
  cp "$SAVED" /tmp/llama-bin/llama-server && chmod +x /tmp/llama-bin/llama-server
  echo "reusing saved build: $SAVED (skipping the ~30 min compile)"
  /tmp/llama-bin/llama-server --version 2>&1 | tail -2
  exit 0
fi

cd /tmp
[ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
rm -rf llama.cpp/build   # a failed configure leaves a bad CMakeCache behind

# Point CMake at the real CUDA toolkit and libcuda explicitly (Kaggle images don't expose CUDA::cuda_driver)
NVCC=$(ls /usr/local/cuda/bin/nvcc 2>/dev/null || command -v nvcc)
CUDA_HOME=$(dirname "$(dirname "$(readlink -f "$NVCC")")")
LIBCUDA=$(ls "$CUDA_HOME"/lib64/stubs/libcuda.so "$CUDA_HOME"/targets/x86_64-linux/lib/stubs/libcuda.so \
             /usr/lib/x86_64-linux-gnu/libcuda.so* /usr/local/nvidia/lib64/libcuda.so* 2>/dev/null | head -1 || true)
echo "nvcc:    $NVCC"
echo "toolkit: $CUDA_HOME"
echo "libcuda: ${LIBCUDA:-NOT FOUND}"

# BUILD_SHARED_LIBS=OFF: one self-contained llama-server binary, easy to save and reuse
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF \
      -DBUILD_SHARED_LIBS=OFF -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_COMPILER="$NVCC" -DCUDAToolkit_ROOT="$CUDA_HOME" \
      ${LIBCUDA:+-DCUDA_cuda_driver_LIBRARY="$LIBCUDA"} > /tmp/cmake.log 2>&1 \
  || { echo "--- cmake failed, last lines:"; tail -40 /tmp/cmake.log; exit 1; }
echo "configured, building (~30 min on Kaggle's 4 CPUs)..."
cmake --build llama.cpp/build -j"$(nproc)" --target llama-server > /tmp/build.log 2>&1 \
  || { echo "--- build failed, last lines:"; tail -40 /tmp/build.log; exit 1; }

cp llama.cpp/build/bin/llama-server /tmp/llama-bin/llama-server
mkdir -p /kaggle/working/llama-bin && cp llama.cpp/build/bin/llama-server /kaggle/working/llama-bin/
ls -la /tmp/llama-bin/llama-server
echo "Saved to /kaggle/working/llama-bin. Use 'Save Version' so the next session can skip this build."

## 2. Download Evon (to /tmp: /kaggle/working is limited to ~20 GB)

In [ ]:
from huggingface_hub import hf_hub_download
MODEL = hf_hub_download("luminenio/gnani-evon-v3.3-30B-A3B-GGUF",
                        "gnani-evon-v3.3-30B-A3B-IQ4_XS.gguf", local_dir="/tmp/models")
print(MODEL)

## 3. Start the server (split across both GPUs, API key required)

In [ ]:
import secrets, subprocess, time, urllib.request
EVON_API_KEY = secrets.token_urlsafe(24)
server = subprocess.Popen(
    ["/tmp/llama-bin/llama-server", "-m", MODEL, "--host", "127.0.0.1", "--port", "8080",
     "-ngl", "99", "--ctx-size", "8192", "--jinja", "--api-key", EVON_API_KEY, "--alias", "evon"],
    stdout=open("/tmp/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8080/health").status == 200:
            print("server ready"); break
    except Exception:
        time.sleep(5)
else:
    print(open("/tmp/server.log").read()[-3000:])

## 4. Quick test inside Kaggle

In [ ]:
import json, time, urllib.request
def ask(text):
    body = {"model": "evon", "temperature": 0, "max_tokens": 300,
            "chat_template_kwargs": {"enable_thinking": False},
            "messages": [{"role": "user", "content": "Is this phone call a scam? Answer in one JSON object "
                          '{"risk": 0-100, "reason": "..."}. Call: ' + text}]}
    req = urllib.request.Request("http://127.0.0.1:8080/v1/chat/completions", json.dumps(body).encode(),
                                 {"Content-Type": "application/json", "Authorization": f"Bearer {EVON_API_KEY}"})
    t0 = time.time()
    out = json.load(urllib.request.urlopen(req))["choices"][0]["message"]["content"]
    print(f"{time.time() - t0:.1f}s", out)
ask("Main CBI se bol raha hoon, aapke naam pe arrest warrant hai, camera on rakhiye.")
ask("Namaste aunty, kal subah gyarah baje aapka checkup hai.")

## 5. Public HTTPS URL via Cloudflare quick tunnel

In [ ]:
import re, subprocess, time
subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
               "-O /tmp/cloudflared && chmod +x /tmp/cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8080", "--no-autoupdate"],
                          stdout=open("/tmp/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/tunnel.log").read())
    if m:
        url = m.group(0); break
    time.sleep(2)
print("Put these two lines in kavach/.env, then restart the Kavach server:\n")
print(f"EVON_URL={url}/v1/chat/completions")
print(f"EVON_API_KEY={EVON_API_KEY}")

Keep this notebook open while you use Kavach. If the tunnel URL stops working, re-run step 5.